## The Sporting Director: Data Pipeline

**Objective:** Build a clean, normalized player database from Kaggle databases (FBref match statics and Transfermarkt valuations) to feed the recommendation algorithm.

**Pipeline Flow:**
1. **Merging & Cleansing:** Load and merge raw CSV files from Kaggle. Standardize player/club names and compute a time-weighted average for seasonal metrics (60% last season, 40% previous).
2. **Data Normalization:** Apply Min-Max scaling to statistical metrics to ensure equal weighting scales.
3. **Multi-Criteria Rating:** Calculate a final performance score based on positional roles, adjusted by age and reliability (minutes played).
4. **Output:** Export the finalized dataset to SQLite (`players.db`).

**Data Sources:**
* **Kaggle Datasets:** 
  * `players.csv` & `player_valuations.csv` ("Football Data from Transfermarkt" by davidcariboo. You can find the original data [here](https://www.kaggle.com/datasets/davidcariboo/player-scores)).
  * `players_stats_2024_2025.csv` ("Football Players Stats (2025-2026)" by hubertsidorowicz. You can find the original data [here](https://www.kaggle.com/datasets/hubertsidorowicz/football-players-stats-2025-2026)).
  * `players_stats_2025_2026.csv` ("Football Players Stats (2024-2025)" by hubertsidorowicz. You can find the original data [here](https://www.kaggle.com/datasets/hubertsidorowicz/football-players-stats-2024-2025)).



In [1]:
import pandas as pd
import unicodedata
import sqlite3
import re

# Load raw datasets
df_players = pd.read_csv("data/players.csv")
df_valuations = pd.read_csv("data/player_valuations.csv")
df_stats_26 = pd.read_csv("data/players_stats_2025_2026.csv") # Season 2025-2026
df_stats_25 = pd.read_csv("data/players_stats_2024_2025.csv") # Season 2024-2025

### 1. Merging and cleansing

Merge `players.csv` and `player_valuations.csv` in order to obtain the most recent market value and an weighted average of stats in a single dataset.


In [2]:
# Order market value record by date (from the most recent to the oldest)
df_valuations = df_valuations.sort_values(by=["player_id", "date"], ascending=[True, False])

# Remove duplicates, the lastest price remains.
df_lastest_valuations = df_valuations.drop_duplicates(subset="player_id", keep="first")

# Merge players table with their most recent value
df_market = pd.merge(df_players, df_lastest_valuations, on="player_id", how="left")

In [3]:
# Data cleansing

# Player names
def name_cleansing(name):
    # Normalize player names by removing accents and converting to lowercase.
    if pd.isna(name):
        return ""
    name = str(name).lower().strip()
    name = (
    name.replace("ø", "o")
        .replace("æ", "ae")
        .replace("ß", "ss")) 
    name = ''.join(c for c in unicodedata.normalize('NFD', name) if unicodedata.category(c) != 'Mn')
    return name

# Club names
def club_cleansing(club):
    # Standardize club names by removing common suffixes and normalizing text.
    if pd.isna(club):
        return ""
    club = str(club).lower().strip()
    club = ''.join(c for c in unicodedata.normalize('NFD', club) if unicodedata.category(c) != 'Mn')

    # Remove regular prefixes and suffixes in football
    club = re.sub(r"\b(fc|fsv|cf|cfc|acf|losc|aj|rc|ud|bc|sc|vfb|vfl|us|ss|sc|ssc|rcd|afc)\b", "", club)
    club = re.sub(r"\s+", " ", club).strip()

    # Fix some differences between datasets values
    replacements = {
        "utd": "united",
    }
    for old, new in replacements.items():
            club = club.replace(old, new).strip()

    club_mapping = {
        "strasbourg alsace":"strasbourg",
        "stade rennais": "rennes",
        "marseille": "olympique marseille",
        "lyon": "olympique lyon",
        "atletico madrid": "atletico de madrid",
        "athletic bilbao": "athletic club",
        "milan": "ac milan",
        "le havre ac": "le havre",
        "wolverhampton wanderers": "wolves",
        "deportivo alaves": "alaves",
        "celta vigo": "celta de vigo",
        "espanyol barcelona": "espanyol",
        "dortmund": "borussia dortmund",
        "gladbach": "borussia mönchengladbach",
        "leverkusen": "bayer 04 leverkusen" ,
        "calcio como": "como",
        "roma": "as roma",
        "as monaco": "monaco",
        "valladolid": "real valladolid",
        "oviedo": "real oviedo",
        "real betis": "real betis balompie",
        "ca osasuna": "osasuna",
        "brighton & hove albion": "brighton",
        "internazionale": "inter"
    }

    return club_mapping.get(club, club)

# Apply the cleaning to the market dataframe
df_market["player_name"] = df_market["name"].apply(name_cleansing)
df_market["club"] = df_market["current_club_name_x"].apply(club_cleansing)

In [4]:
# Merge the stats tables for the two seasons into a single dataframe

# Clean names in statistics tables to prepare for cross-season merging
name_26 = pd.DataFrame({"player_name": df_stats_26["Player"].apply(name_cleansing)})
name_25 = pd.DataFrame({"player_name": df_stats_25["Player"].apply(name_cleansing)})

df_stats_26 = pd.concat([df_stats_26, name_26], axis=1)
df_stats_25 = pd.concat([df_stats_25, name_25], axis=1)

# Define the columns to exclude from the merging process
exclude = {"player_name", "Nation", "Pos", "Squad", "Comp", "Age", "Born"}
cols_26 = set(df_stats_26.select_dtypes(include=["number"]).columns) - exclude
cols_25 = set(df_stats_25.select_dtypes(include=["number"]).columns) - exclude

# Identify metrics present in both seasons to calculate weighted averages
simultaneous_cols = list(cols_26.intersection(cols_25))
# Outer join stats from both seasons
df_stats = pd.merge(df_stats_26, df_stats_25, on="player_name", how="outer", suffixes=("_26", "_25"))

# Create new values for the simultaneous columns
for col in simultaneous_cols:
    col_26 = f"{col}_26"
    col_25 = f"{col}_25"

    if col_26 in df_stats.columns and col_25 in df_stats.columns:
        # If the column exists in both seasons, we take a weighted average of the two values, giving more weight to the most recent season.
        df_stats[col] = df_stats[col_26].fillna(0) * 0.6 + df_stats[col_25].fillna(0) * 0.4
    elif col_26 in df_stats.columns:
        # If the column exists only in the most recent season, we take that value.
        df_stats[col] = df_stats[col_26].fillna(0)
    elif col_25 in df_stats.columns:
        # If the column exists only in the previous season, we take that value.
        df_stats[col] = df_stats[col_25].fillna(0)

new_cols = pd.DataFrame({"club": df_stats["Squad_26"].fillna(df_stats["Squad_25"]).apply(club_cleansing),
                         "Pos": df_stats["Pos_26"].fillna(df_stats["Pos_25"]),
                         "Age": df_stats["Age_26"].fillna(df_stats["Age_25"] + 1)}) 

df_stats = pd.concat([df_stats, new_cols], axis=1)

C:\Users\andre\AppData\Local\Temp\ipykernel_25280\3965685783.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_stats[col] = df_stats[col_26].fillna(0) * 0.6 + df_stats[col_25].fillna(0) * 0.4


In [5]:
# Final merge combining statistical data with market valuations
df = pd.merge(df_stats, df_market, on=["player_name", "club"], how="left")
print(f"players loaded: {len(df)}")

# Quality of the merge
match_rate = df["market_value_in_eur_y"].notna().mean()
print(f"Players matched: {match_rate:.1%}")
print(f"Players unmatched: {1-match_rate:.1%}")

players loaded: 3935
Players matched: 50.4%
Players unmatched: 49.6%


### 2. Data Normalization

To compare heterogeneous metrics on a single scale, mathematical normalization is applied. This applies to purely statistical columns (goals, key passes, tackles won).
Each variable is transformed into a range [0,1] using the min-max normalization:

$$X_{norm} = \frac{X - X_{min}}{X_{max} - X_{min}}$$


In [6]:
# Normalize numeric columns (the most interesting)
cols = ["Save%", "CS%", "PKsv", "GA",
        "TklW", "Int", "Clr", "Err", "CrdR", "2CrdY",
        "G+A", "KP", "PrgP", "PrgC", "Cmp%", "Mis",
        "Gls", "G+A-PK", "Carries", "xG", "90s"]

existing_cols = [col for col in cols if col in df.columns]

# Vectorized scaling to map metrics to a [0, 1] interval safely
df_min = df[existing_cols].min()
df_max = df[existing_cols].max()
df_range = (df_max -df_min).replace(0, 1) # Avoid division by zero

norm_cols = [f"{col}_norm" for col in existing_cols]
df[norm_cols] = (df[existing_cols] - df_min) / df_range

C:\Users\andre\AppData\Local\Temp\ipykernel_25280\2538887654.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[norm_cols] = (df[existing_cols] - df_min) / df_range


### 3. Multi-Criteria Rating

Assign weighted rating to each player's on field position, prioritizing real-impact metrics.

* **Forwards**(FW): Prioritizes scoring capacity and offensive danger.
* **Midfielders**(MF): Values playmaking and distribution (G+A, KP, PrgP, PrgC).
* **Defenders**(DF) and Goalkeepers (GK): Focuses on clean defensive actions, save percentages, and error prevention.

Moreover, we penalize veterans (likely to injuries) and award a bonus to peroyoung players (potential growth).
Finally, the score is multiplied by the normalised completed matches played. This acts as a filter that penalises players with insufficient data samples (low participation or long-term injuries).

In [7]:
# Position weights
position_weights = {
    "GK": {"Save%_norm": 0.55, "CS%_norm": 0.35, "PKsv_norm": 0.10, "GA_norm": -0.20},
    "DF": {"TklW_norm": 0.30, "Int_norm": 0.25, "Clr_norm": 0.25, "PrgP_norm": 0.15, "CS%_norm": 0.05, "Err_norm": -0.20, "CrdR_norm": -0.10},
    "MF": {"G+A_norm": 0.25, "KP_norm": 0.25, "PrgP_norm": 0.20, "PrgC_norm": 0.20, "Cmp%_norm": 0.10, "Mis_norm": -0.10},
    "FW": {"Gls_norm": 0.40, "xG_norm": 0.20, "G+A-PK_norm": 0.15, "Carries_norm": 0.15, "PrgP_norm": 0.10, "Mis_norm": -0.10}
}

def rating(row):
    # Calculate dynamic player score based on role weights, age modifiers, and reliability
    pos = str(row["Pos"]).upper()
    fiability = row.get("90s_norm", 0)
    score = 0.0
    
    # Calculate score based on position mapping dynamically
    for key, weights in position_weights.items():
        if key in pos:
            score = sum(row.get(col, 0) * weight for col, weight in weights.items())
            break
            
    # Penalize veterans (injury risk) and reward young players (potential growth)
    age = row["Age"]
    if age <= 25:
        score *= 1.1
    elif age > 33:
        score *= 0.9
        
    # Multiplied by normalized completed matches played (penalizes low participation/injuries)
    return score * fiability

# Apply rating evaluation per row
df["rating"] = df.apply(rating, axis=1)

C:\Users\andre\AppData\Local\Temp\ipykernel_25280\2701796881.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["rating"] = df.apply(rating, axis=1)


### 4. Output: Export SQL database

In [8]:
connect = sqlite3.connect("players.db")
df.to_sql("players", connect, if_exists="replace", index=False)
connect.close()
print("Data has been successfully saved to the SQLite database 'players.db'.")

Data has been successfully saved to the SQLite database 'players.db'.
